# Week 4 Lab — Structured output, streaming, cost tracking, model comparison
Run from the project root with the `.venv` kernel. The key is read from `.env` (`OPENAI_API_KEY`); never paste it into a cell. Code lives in `src/capstone/` (`cost.py`, `llm.py`, `rag.py`, `schemas.py`); this notebook exercises it.

In [ ]:
import os, sys, asyncio, json, time
os.chdir('/Users/hvt/Downloads/capstone'); sys.path.insert(0, 'src')
from dotenv import load_dotenv; load_dotenv(override=True)
os.environ['BACKEND'] = 'openai'
assert os.getenv('OPENAI_API_KEY'), 'put OPENAI_API_KEY in .env'
print('key loaded:', os.environ['OPENAI_API_KEY'][:7] + '…')

## 1. Offline tests (fake backend)

In [ ]:
!BACKEND=fake .venv/bin/python -m pytest -q

## 2. Boot pipeline

In [ ]:
from capstone.config import get_settings
from capstone.llm import LLM, Cache
from capstone.retrieve import Retriever
from capstone.store import Store
from capstone import rag
s = get_settings()
chunks, m = Store(s.db_path).load()
llm = LLM(s, Cache(s.db_path))
retriever = Retriever(chunks, m, llm)
print(s.chat_model, len(chunks), 'chunks')

## 3. Structured output via tool-calling (schema v1)
`submit_answer` is forced; arguments are validated by pydantic into `schemas.Answer`.

In [ ]:
from capstone.schemas import Answer, AskResponse, Usage
a = await rag.answer('How many days of annual leave do I get per year?', llm, retriever, s)
print(AskResponse(answer=a.text, citations=a.citations, refused=a.refused, usage=Usage(**a.usage)).model_dump_json(indent=2))

In [ ]:
# schema_version is a Literal: anything but 'v1' is rejected
from pydantic import ValidationError
try: Answer(answer='x', schema_version='v2')
except ValidationError as e: print('rejected:', e.errors()[0]['msg'])

## 4. Real streaming: time-to-first-token, total latency, usage-based cost

In [ ]:
t0 = time.monotonic(); first = None
async for kind, val in rag.answer_stream('When do I need a medical certificate for sick leave?', llm, retriever, s):
    if kind == 'token':
        if first is None: first = time.monotonic() - t0
        print(val, end='', flush=True)
    else: fin = val
print(f"\n\nTTFT {first:.2f}s · total {time.monotonic()-t0:.2f}s · {fin.usage}")

## 5. Batch with persisted model + cost

In [ ]:
from capstone.batch import run_batch
qs = [l.strip() for l in open('eval/questions.txt') if l.strip()]
res = await run_batch(qs, llm, retriever, s)
print(sum(r.status=='ok' for r in res), '/', len(res), 'ok · batch cost $%.6f' % sum(r.cost_usd for r in res))
print(res[0].model_dump())

In [ ]:
import sqlite3
c = sqlite3.connect(s.db_path)
print(c.execute('SELECT model, COUNT(*), SUM(prompt_tokens), SUM(completion_tokens), ROUND(SUM(cost_usd),6) FROM llm_usage GROUP BY model').fetchall())
print('session stats:', llm.stats)

## 6. Compare two models (quality / cost / latency)
Writes `docs/model_comparison.md`.

In [ ]:
!PYTHONPATH=src BACKEND=openai .venv/bin/python scripts/compare_models.py gpt-4o-mini gpt-4o

**Analysis:** see the Conclusion in `docs/model_comparison.md`; decisions recorded in `docs/adr/0004-structured-output-cost-and-schema-versioning.md`.